In [51]:
!pip install numpy matplotlib seaborn
!pip install scikit-learn

# Data manipulation
import pandas as pd
import numpy as np

# Vizualization
import matplotlib.pyplot as plt
import seaborn as sb

# Machine learning
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import GridSearchCV

# Warnings
import warnings
warnings.filterwarnings('ignore')

### Exploring the data

In [29]:
sales = pd.read_csv("stores_sales_forecasting.csv", encoding='latin1')

In [31]:
# Lines and columns
sales.shape

(2121, 21)

In [38]:
sales.size

44541

In [25]:
sales.head()

,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,...,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit
0,1,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.9600,2,0.00,41.9136
1,2,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.9400,3,0.00,219.5820
2,4,US-2015-108966,10/11/2015,10/18/2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775,5,0.45,-383.0310
3,6,CA-2014-115812,6/9/2014,6/14/2014,Standard Class,BH-11710,Brosina Hoffman,Consumer,United States,Los Angeles,...,90032,West,FUR-FU-10001487,Furniture,Furnishings,Eldon Expressions Wood and Plastic Desk Access...,48.8600,7,0.00,14.1694
4,11,CA-2014-115812,6/9/2014,6/14/2014,Standard Class,BH-11710,Brosina Hoffman,Consumer,United States,Los Angeles,...,90032,West,FUR-TA-10001539,Furniture,Tables,Chromcraft Rectangular Conference Tables,1706.1840,9,0.20,85.3092


In [34]:
# Checking for total null values
sales.isnull().sum()

Row ID           0
Order ID         0
Order Date       0
Ship Date        0
Ship Mode        0
Customer ID      0
Customer Name    0
Segment          0
Country          0
City             0
State            0
Postal Code      0
Region           0
Product ID       0
Category         0
Sub-Category     0
Product Name     0
Sales            0
Quantity         0
Discount         0
Profit           0
dtype: int64

In [35]:
sales.dtypes

Row ID             int64
Order ID          object
Order Date        object
Ship Date         object
Ship Mode         object
Customer ID       object
Customer Name     object
Segment           object
Country           object
City              object
State             object
Postal Code        int64
Region            object
Product ID        object
Category          object
Sub-Category      object
Product Name      object
Sales            float64
Quantity           int64
Discount         float64
Profit           float64
dtype: object

In [36]:
sales.describe()

,Row ID,Postal Code,Sales,Quantity,Discount,Profit
count,2121.000000,2121.000000,2121.000000,2121.000000,2121.000000,2121.000000
mean,5041.643564,55726.556341,349.834887,3.785007,0.173923,8.699327
std,2885.740258,32261.888225,503.179145,2.251620,0.181547,136.049246
min,1.000000,1040.000000,1.892000,1.000000,0.000000,-1862.312400
25%,2568.000000,22801.000000,47.040000,2.000000,0.000000,-12.849000
50%,5145.000000,60505.000000,182.220000,3.000000,0.200000,7.774800
75%,7534.000000,90032.000000,435.168000,5.000000,0.300000,33.726600
max,9991.000000,99301.000000,4416.174000,14.000000,0.700000,1013.127000


### Data Cleaning

In [31]:
sales.drop(['Row ID', 'Postal Code', 'Region'], axis=1, inplace = True)

In [32]:
sales.head()

,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,State,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit
0,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,Kentucky,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.9600,2,0.00,41.9136
1,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,Kentucky,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.9400,3,0.00,219.5820
2,US-2015-108966,10/11/2015,10/18/2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,Florida,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775,5,0.45,-383.0310
3,CA-2014-115812,6/9/2014,6/14/2014,Standard Class,BH-11710,Brosina Hoffman,Consumer,United States,Los Angeles,California,FUR-FU-10001487,Furniture,Furnishings,Eldon Expressions Wood and Plastic Desk Access...,48.8600,7,0.00,14.1694
4,CA-2014-115812,6/9/2014,6/14/2014,Standard Class,BH-11710,Brosina Hoffman,Consumer,United States,Los Angeles,California,FUR-TA-10001539,Furniture,Tables,Chromcraft Rectangular Conference Tables,1706.1840,9,0.20,85.3092


In [65]:
sales.Country.value_counts()

Country
United States    2121
Name: count, dtype: int64

In [36]:
# Convert date columns
sales['Order Date'] = pd.to_datetime(sales['Order Date'])
sales['Ship Date'] = pd.to_datetime(sales['Ship Date'])

In [17]:
print(sales.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2121 entries, 0 to 2120
Data columns (total 21 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   Order Date         2121 non-null   datetime64[ns]
 1   Ship Date          2121 non-null   datetime64[ns]
 2   Ship Mode          2121 non-null   int64         
 3   Customer ID        2121 non-null   object        
 4   Customer Name      2121 non-null   object        
 5   Segment            2121 non-null   int64         
 6   Country            2121 non-null   object        
 7   City               2121 non-null   int64         
 8   State              2121 non-null   int64         
 9   Product ID         2121 non-null   object        
 10  Category           2121 non-null   int64         
 11  Product Name       2121 non-null   int64         
 12  Sales              2121 non-null   float64       
 13  Quantity           2121 non-null   int64         
 14  Discount

### Feature Engineering

#### Add additional features

In [37]:
# Extract features from date column
sales['Order Month'] = sales['Order Date'].dt.month
sales['Order Year'] = sales['Order Date'].dt.year
sales['Order Day'] = sales['Order Date'].dt.day
sales['Order Weekday'] = sales['Order Date'].dt.weekday
sales['Quarter'] = sales['Order Date'].dt.quarter

# Calculate shipping time (days)
sales['Shipping Time'] = (sales['Ship Date'] - sales['Order Date']).dt.days

# Encode categorical features
categorical_cols = ['Ship Mode', 'Segment', 'Category', 'City', 'State', 'Product Name']

label_encoders = {}
for col in categorical_cols:
    label_encoders[col] = LabelEncoder()
    sales[col] = label_encoders[col].fit_transform(sales[col])

In [38]:
sales.head()

,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,State,...,Sales,Quantity,Discount,Profit,Order Month,Order Year,Order Day,Order Weekday,Quarter,Shipping Time
0,CA-2016-152156,2016-11-08,2016-11-11,2,CG-12520,Claire Gute,0,United States,137,15,...,261.9600,2,0.00,41.9136,11,2016,8,1,4,3
1,CA-2016-152156,2016-11-08,2016-11-11,2,CG-12520,Claire Gute,0,United States,137,15,...,731.9400,3,0.00,219.5820,11,2016,8,1,4,3
2,US-2015-108966,2015-10-11,2015-10-18,3,SO-20335,Sean O'Donnell,0,United States,108,8,...,957.5775,5,0.45,-383.0310,10,2015,11,6,4,7
3,CA-2014-115812,2014-06-09,2014-06-14,3,BH-11710,Brosina Hoffman,0,United States,184,3,...,48.8600,7,0.00,14.1694,6,2014,9,0,2,5
4,CA-2014-115812,2014-06-09,2014-06-14,3,BH-11710,Brosina Hoffman,0,United States,184,3,...,1706.1840,9,0.20,85.3092,6,2014,9,0,2,5


#### Aggregate data by month

In [124]:
# Create a new column
sales['Month Year'] = sales['Order Date'].dt.to_period('M')

# Aggregate data
monthly_sales = sales.groupby('Month Year').agg({'Sales': 'sum'}).reset_index()
monthly_sales.head()

,Month Year,Sales
0,2014-01,6242.525
1,2014-02,1839.658
2,2014-03,14573.956
3,2014-04,7944.837
4,2014-05,6912.787


In [114]:
monthly_sales

,Month Year,Sales,sales_lag_1,sales_lag_2,sales_lag_3,rolling_avg_3,rolling_avg_6
8,2014-09,23816.4808,7320.3465,10821.0510,13206.1256,13985.959433,11670.271317
9,2014-10,12304.2470,23816.4808,7320.3465,10821.0510,14480.358100,12396.839650
10,2014-11,21564.8727,12304.2470,23816.4808,7320.3465,19228.533500,14838.853933
11,2014-12,30645.9665,21564.8727,12304.2470,23816.4808,21505.028733,17745.494083
12,2015-01,11739.9416,30645.9665,21564.8727,12304.2470,21316.926933,17898.642517
13,2015-02,3134.3740,11739.9416,30645.9665,21564.8727,15173.427367,17200.980433
14,2015-03,12499.7830,3134.3740,11739.9416,30645.9665,9124.699533,15314.864133
15,2015-04,10475.6985,12499.7830,3134.3740,11739.9416,8703.285167,15010.106050
16,2015-05,9374.9505,10475.6985,12499.7830,3134.3740,10783.477333,12978.452350
17,2015-06,7714.1790,9374.9505,10475.6985,12499.7830,9188.276000,9156.487767


#### Lag Features

In [126]:
monthly_sales

,Month Year,Sales
0,2014-01,6242.5250
1,2014-02,1839.6580
2,2014-03,14573.9560
3,2014-04,7944.8370
4,2014-05,6912.7870
5,2014-06,13206.1256
6,2014-07,10821.0510
7,2014-08,7320.3465
8,2014-09,23816.4808
9,2014-10,12304.2470


In [127]:
monthly_sales['sales_lag_1'] = monthly_sales['Sales'].shift(1)
monthly_sales['sales_lag_2'] = monthly_sales['Sales'].shift(2)
monthly_sales['sales_lag_3'] = monthly_sales['Sales'].shift(3)

# Drop missing values if any
monthly_sales.head()

,Month Year,Sales,sales_lag_1,sales_lag_2,sales_lag_3
0,2014-01,6242.525,NaN,NaN,NaN
1,2014-02,1839.658,6242.525,NaN,NaN
2,2014-03,14573.956,1839.658,6242.525,NaN
3,2014-04,7944.837,14573.956,1839.658,6242.525
4,2014-05,6912.787,7944.837,14573.956,1839.658


#### Rolling window features

In [130]:
# Rolling averages
monthly_sales['rolling_avg_3'] = monthly_sales['Sales'].rolling(window=3).mean()
monthly_sales['rolling_avg_6'] = monthly_sales['Sales'].rolling(window=6).mean()

# Drop missing values after rolling
monthly_sales = monthly_sales.dropna()
monthly_sales

,Month Year,Sales,sales_lag_1,sales_lag_2,sales_lag_3,rolling_avg_3,rolling_avg_6
5,2014-06,13206.1256,6912.7870,7944.8370,14573.9560,9354.583200,8453.314767
6,2014-07,10821.0510,13206.1256,6912.7870,7944.8370,10313.321200,9216.402433
7,2014-08,7320.3465,10821.0510,13206.1256,6912.7870,10449.174367,10129.850517
8,2014-09,23816.4808,7320.3465,10821.0510,13206.1256,13985.959433,11670.271317
9,2014-10,12304.2470,23816.4808,7320.3465,10821.0510,14480.358100,12396.839650
10,2014-11,21564.8727,12304.2470,23816.4808,7320.3465,19228.533500,14838.853933
11,2014-12,30645.9665,21564.8727,12304.2470,23816.4808,21505.028733,17745.494083
12,2015-01,11739.9416,30645.9665,21564.8727,12304.2470,21316.926933,17898.642517
13,2015-02,3134.3740,11739.9416,30645.9665,21564.8727,15173.427367,17200.980433
14,2015-03,12499.7830,3134.3740,11739.9416,30645.9665,9124.699533,15314.864133


In [131]:
monthly_sales.info()

<class 'pandas.core.frame.DataFrame'>
Index: 43 entries, 5 to 47
Data columns (total 7 columns):
 #   Column         Non-Null Count  Dtype    
---  ------         --------------  -----    
 0   Month Year     43 non-null     period[M]
 1   Sales          43 non-null     float64  
 2   sales_lag_1    43 non-null     float64  
 3   sales_lag_2    43 non-null     float64  
 4   sales_lag_3    43 non-null     float64  
 5   rolling_avg_3  43 non-null     float64  
 6   rolling_avg_6  43 non-null     float64  
dtypes: float64(6), period[M](1)
memory usage: 2.7 KB


### Define Target and Features

In [132]:
# Separate target variable (Sales) from the predictors
target = 'Sales'
features = ['sales_lag_1', 'sales_lag_2', 'sales_lag_3', 'rolling_avg_3', 'rolling_avg_6']

X = monthly_sales[features]
y = monthly_sales[target]

### Split the data into training and testing sets

In [133]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False)

print(f"Training set: {X_train.shape}")
print(f"Testing set: {X_test.shape}")

Training set: (34, 5)
Testing set: (9, 5)


In [134]:
X_test

,sales_lag_1,sales_lag_2,sales_lag_3,rolling_avg_3,rolling_avg_6
39,10893.4448,6866.3374,5964.0320,8941.913433,16875.352683
40,9065.9581,10893.4448,6866.3374,12305.653700,14404.340917
41,16957.5582,9065.9581,10893.4448,15010.701000,11459.319533
42,19008.5867,16957.5582,9065.9581,15926.388967,12434.151200
43,11813.0220,19008.5867,16957.5582,15421.160900,13863.407300
44,15441.8740,11813.0220,19008.5867,18761.034000,16885.867500
45,29028.2060,15441.8740,11813.0220,22118.049400,19022.219183
46,21884.0682,29028.2060,15441.8740,29322.996400,22372.078650
47,37056.7150,21884.0682,29028.2060,30116.083333,24438.558667


In [135]:
y_test

39     9065.9581
40    16957.5582
41    19008.5867
42    11813.0220
43    15441.8740
44    29028.2060
45    21884.0682
46    37056.7150
47    31407.4668
Name: Sales, dtype: float64

### Train a Random Forest Regressor

In [148]:
# Initialize the model
rf_model = RandomForestRegressor(random_state=42, n_estimators=165, max_depth=10, min_samples_split=3)

# Train the model
rf_model.fit(X_train, y_train)

RandomForestRegressor(max_depth=10, min_samples_split=3, n_estimators=165,
                      random_state=42)

### Evaluate the Model

In [149]:
# Make predictions
y_pred = rf_model.predict(X_test)

# Evaluate performance

# MAE measures the average of the absolute differences between the predicted values and the actual values (in this case 128$ which is BAD)
mae = mean_absolute_error(y_test, y_pred)
# MSE
mse = mean_squared_error(y_test, y_pred)
# RMSE similar to MAE but gives more weight to larger errors
rmse = np.sqrt(mse)

print(f"Mean Absolute Error (MAE): {mae}")
print(f"Mean Squared Error (MSE): {mse}")
print(f"Root Mean Squared Error (RMSE): {rmse}")

Mean Absolute Error (MAE): 4127.835147865971
Mean Squared Error (MSE): 25165962.645726707
Root Mean Squared Error (RMSE): 5016.568812019497


#### Tune Hyperparameters

In [138]:
# Define parameter grid
param_grid = {
    'n_estimators': [140, 150, 160],
    'max_depth': [9, 10, 11],
    'min_samples_split': [1, 2, 3]
}

# Create GridSearchCV
grid_search = GridSearchCV(estimator=RandomForestRegressor(), param_grid=param_grid, cv=5)

# Fit grid search
grid_search.fit(X_train, y_train)

# Get the best parameters
best_params = grid_search.best_params_
print(f"Best Parameters: {best_params}")

Best Parameters: {'max_depth': 10, 'min_samples_split': 3, 'n_estimators': 160}


### Future Predictions

In [151]:
# The most recent actual data
latest_actual_sales = monthly_sales.iloc[-1]['Sales']
latest_data = X_test.iloc[-1].copy()

# Replace lagged features
latest_data['sales_lag_1'] = latest_actual_sales
latest_data['sales_lag_2'] = monthly_sales.iloc[-2]['Sales']
latest_data['sales_lag_3'] = monthly_sales.iloc[-3]['Sales']
latest_data['rolling_avg_3'] = monthly_sales['Sales'][-3:].mean()
latest_data['rolling_avg_6'] = monthly_sales['Sales'][-6:].mean()
monthly_sales_copy = monthly_sales.copy()

# Empty list to store forecast values
forecast = []

# Forecast the next 3 months
for i in range(3):
    # predict the next month's sales
    predicted_sales = rf_model.predict(latest_data.values.reshape(1, -1))
    next_month = monthly_sales_copy['Month Year'].max() + 1
    forecast.append({'Month Year': next_month, 'Sales': predicted_sales})

    # append the forecasted data to the initial dataset
    new_row = pd.DataFrame({'Month Year': [next_month], 'Sales': [predicted_sales]})
    monthly_sales_copy = pd.concat([monthly_sales_copy, new_row], ignore_index=True)

    # update tha lag features for the next month
    latest_data['sales_lag_3'] = latest_data['sales_lag_2']
    latest_data['sales_lag_2'] = latest_data['sales_lag_1']
    latest_data['sales_lag_1'] = predicted_sales

    # update the rolling window features
    latest_data['rolling_avg_3'] = monthly_sales_copy['Sales'][-3:].mean()
    latest_data['rolling_avg_6'] = monthly_sales_copy['Sales'][-6:].mean()

    # append the forecasted value to the test set for the next prediction
    # X_test = X_test.append(latest_data)
forecast_df = pd.DataFrame(forecast)
# Show the forecast for the next 3 months
print(f"Forecasted sales for the next 3 months: {forecast_df}")


Forecasted sales for the next 3 months:   Month Year                 Sales
0    2018-01   [24481.69530597112]
1    2018-02  [24832.752535769094]
2    2018-03  [30045.386034747422]
